# Lesson 2 Demo Notebook: Stateless and Stateful Agents

This notebook gives you a single flow for live teaching of Lesson 2.

Sections:
1. Setup
2. Stateless call (forgets)
3. Stateful call (remembers)
4. Measure token growth
5. Windowing concept demo
6. Optional: run full CLI scripts


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env")

client = OpenAI()
MODEL = "gpt-4.1-mini"
print("Client ready")


## 1) Stateless: each turn is independent

We intentionally *do not* send prior messages on turn 2.

In [ ]:
turn1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Alice."}]
)
print("Turn 1:", turn1.choices[0].message.content)

turn2 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is my name?"}]
)
print("Turn 2:", turn2.choices[0].message.content)


## 2) Stateful: send history every turn

Now we carry message history forward manually.

In [ ]:
history = []

history.append({"role": "user", "content": "My name is Alice."})
r1 = client.chat.completions.create(model=MODEL, messages=history)
a1 = r1.choices[0].message.content
history.append({"role": "assistant", "content": a1})

history.append({"role": "user", "content": "What is my name?"})
r2 = client.chat.completions.create(model=MODEL, messages=history)
a2 = r2.choices[0].message.content

print("Assistant turn 1:", a1)
print("Assistant turn 2:", a2)
print("Prompt tokens on turn 2:", r2.usage.prompt_tokens)


## 3) Measure token growth

This approximates the quadratic cost trend in full-history chat.

In [ ]:
prompts = [
    "My name is Alice and I live in Trento.",
    "I work in computer science.",
    "I prefer concise answers.",
    "I am teaching an AI systems class.",
    "Summarize what you know about me so far.",
    "What details might be missing?",
    "Now give me three study tips.",
    "What is my name and where do I live?",
]

history = []
stats = []

for i, p in enumerate(prompts, start=1):
    history.append({"role": "user", "content": p})
    r = client.chat.completions.create(model=MODEL, messages=history, temperature=0.2)
    reply = r.choices[0].message.content
    history.append({"role": "assistant", "content": reply})

    stats.append((i, r.usage.prompt_tokens, r.usage.completion_tokens))

print("turn | prompt_tokens | completion_tokens")
for row in stats:
    print(f"{row[0]:>4} | {row[1]:>13} | {row[2]:>17}")


## 4) Windowing concept (deterministic demo)

This uses the helper logic from `3_agent_with_memory.py` without making extra API calls.

In [ ]:
from importlib.util import spec_from_file_location, module_from_spec
from pathlib import Path

mod_path = Path('labs/02_standalone_agents/3_agent_with_memory.py')
if not mod_path.exists():
    mod_path = Path('../labs/02_standalone_agents/3_agent_with_memory.py')

spec = spec_from_file_location('l2_memory', mod_path)
mod = module_from_spec(spec)
spec.loader.exec_module(mod)

conversation = [
    {"role": "user", "content": "u1"},
    {"role": "assistant", "content": "a1"},
    {"role": "user", "content": "u2"},
    {"role": "assistant", "content": "a2"},
    {"role": "user", "content": "u3"},
    {"role": "assistant", "content": "a3"},
]

window = mod.trim_to_window(conversation, window_turns=2)
print("Original messages:", len(conversation))
print("Kept messages:", len(window))
print(window)

## 5) Optional CLI demos

For full interactive experience (recommended live):
- `uv run python labs/02_standalone_agents/1_stateless_agent.py`
- `uv run python labs/02_standalone_agents/2_stateful_agent.py`
- `uv run python labs/02_standalone_agents/3_agent_with_memory.py`
- `uv run python labs/02_standalone_agents/4_agent_with_long_term_memory.py`